# 03 — Baseline model

Same rule as the other notebooks: **question → code → output → interpretation**, aggregates only, no raw rows.

Features: `src/skeptical_second_reader/features.py`. Model: `src/skeptical_second_reader/model.py`. Folds: notebook 02.

In [1]:
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from skeptical_second_reader.data import (
    ACCOUNTS_CSV,
    PATTERNS_TXT,
    account_entities,
    account_labels,
    load_transactions,
    pattern_members,
    read_accounts,
    read_patterns,
)
from skeptical_second_reader.features import FEATURES, WINDOW_END, account_features
from skeptical_second_reader.model import fit_xgboost
from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.split import (
    N_FOLDS,
    cached_folds,
    cv_predict,
    split_groups,
)

df = load_transactions()
labels = account_labels(df)
owners = account_entities(labels.index, read_accounts(ACCOUNTS_CSV))
members = pattern_members(read_patterns(PATTERNS_TXT))
groups = split_groups(owners, members)
folds = cached_folds(labels, groups, DATA_INTERIM)
print(f"accounts: {labels.size:,}  positives: {labels.sum():,}")

accounts: 515,088  positives: 6,357


## 1. Feature window

**Question:** what does the tail after 10 September reveal, and what does a 1–10 September window cost?

In [2]:
def keys(frame, side):
    return frame[f"{side}_bank"] + ":" + frame[f"{side}_account"]


def accounts_of(frame):
    return pd.Index(pd.concat([keys(frame, "from"), keys(frame, "to")]).unique())


early = df[df["timestamp"] < WINDOW_END]
tail = df[df["timestamp"] >= WINDOW_END]
tail_accounts = accounts_of(tail)
n_tail_pos = labels[tail_accounts].sum()
n_tail_clean = (tail["is_laundering"] == 0).sum()
print(f"tail transactions: {len(tail):,}  clean: {n_tail_clean:,}")
print(f"accounts active in the tail: {tail_accounts.size:,}  positive: {n_tail_pos:,}")

early_positive = accounts_of(early[early["is_laundering"] == 1])
only_tail = labels.index[(labels == 1) & ~labels.index.isin(early_positive)]
early_accounts = accounts_of(early)
absent = only_tail[~only_tail.isin(early_accounts)]
share = only_tail.size / labels.sum()
print(f"positives only through the tail: {only_tail.size:,} ({share:.1%})")
print(f"  with only clean transactions in the window: {only_tail.size - absent.size:,}")
print(f"  with no transaction in the window: {absent.size:,}")
in_patterns = only_tail.isin(members["account"]).sum()
print(f"  in a listed pattern: {in_patterns:,}")

tail transactions: 1,108  clean: 453
accounts active in the tail: 716  positive: 716


positives only through the tail: 495 (7.8%)
  with only clean transactions in the window: 485
  with no transaction in the window: 10
  in a listed pattern: 495


In [3]:
# Payment format inside the window.
by_format = early.groupby("payment_format")["is_laundering"].agg(
    transactions="size", laundering="sum"
)
by_format["share_of_tx_pct"] = (
    100 * by_format["transactions"] / by_format["transactions"].sum()
).round(1)
by_format["share_of_laundering_pct"] = (
    100 * by_format["laundering"] / by_format["laundering"].sum()
).round(1)
by_format["laundering_rate_pct"] = (
    100 * by_format["laundering"] / by_format["transactions"]
).round(3)
by_format.sort_values("laundering", ascending=False)

,transactions,laundering,share_of_tx_pct,share_of_laundering_pct,laundering_rate_pct
payment_format,,,,,
ACH,599689,3828,11.8,84.7,0.638
Cheque,1864331,324,36.7,7.2,0.017
Credit Card,1323324,206,26.1,4.6,0.016
Cash,490891,108,9.7,2.4,0.022
Bitcoin,146091,56,2.9,1.2,0.038
Reinvestment,481056,0,9.5,0.0,0.000
Wire,171855,0,3.4,0.0,0.000


## 2. Features

**Question:** do the features line up with the labels and folds, and are the accounts without a row exactly the expected ones?

In [4]:
%%time
X = account_features(df)
y = labels.loc[X.index]
fold_of = folds.loc[X.index]

assert list(X.columns) == FEATURES
assert set(labels.index) - set(X.index) == set(absent), "unexpected missing accounts"
assert (X.notna()).all().all()
assert ((X["out_tx"] + X["in_tx"]) > 0).all(), "an account without transactions"
assert X["out_tx"].sum() == len(early)
assert X["in_tx"].sum() == len(early)

print(f"model rows: {len(X):,}  positives: {y.sum():,}  rate: {y.mean():.3%}")
print(f"left out: {labels.size - len(X)} accounts, all positive")
print("positives per fold:", y.groupby(fold_of).sum().tolist())

model rows: 515,078  positives: 6,347  rate: 1.232%
left out: 10 accounts, all positive
positives per fold: [1270, 1269, 1269, 1269, 1270]
CPU times: total: 25 s
Wall time: 25 s


In [5]:
# Median and 99th percentile of each feature, for negatives and positives.
summary = pd.DataFrame(
    {
        "neg_median": X[y == 0].median(),
        "pos_median": X[y == 1].median(),
        "neg_p99": X[y == 0].quantile(0.99),
        "pos_p99": X[y == 1].quantile(0.99),
    }
)
summary

,neg_median,pos_median,neg_p99,pos_p99
out_tx,2.0,4.0,90.0,151.00
in_tx,2.0,13.0,49.0,57.00
n_receivers,1.0,2.0,10.0,14.00
n_senders,1.0,3.0,7.0,29.00
active_days,3.0,7.0,10.0,10.00
self_tx,1.0,1.0,5.0,18.54
currency_change_tx,0.0,0.0,2.0,11.54
ach_out_tx,0.0,1.0,17.0,27.54
ach_in_tx,0.0,1.0,14.0,29.00
cheque_tx,1.0,7.0,48.0,70.00


## 3. Baseline: 5-fold cross-validation

**Question:** what AUC-PR does a first XGBoost reach? Two reference points without a model: the positive rate (a random ranking), and ranking accounts by their number of ACH transactions. No threshold yet; AUC-PR needs only the ranking.

In [6]:
%%time
models = []


def fit_and_keep(*args):
    score = fit_xgboost(*args)
    models.append(score.model)
    return score


result = cv_predict(X, y, fold_of, fit_and_keep)

ach_count = X["ach_out_tx"] + X["ach_in_tx"]
rounds = []
for k in range(N_FOLDS):
    test = fold_of == k
    rounds.append(
        {
            "round": k,
            "auc_pr": average_precision_score(y[test], result.scores[test]),
            "auc_roc": roc_auc_score(y[test], result.scores[test]),
            "auc_pr_ach_count": average_precision_score(y[test], ach_count[test]),
            "positive_rate": y[test].mean(),
            "best_iteration": models[k].best_iteration,
        }
    )
rounds = pd.DataFrame(rounds).set_index("round")
rounds.round(4)

CPU times: total: 3min 56s
Wall time: 18.3 s


,auc_pr,auc_roc,auc_pr_ach_count,positive_rate,best_iteration
round,,,,,
0,0.3015,0.8897,0.0303,0.0123,86
1,0.2507,0.8805,0.0378,0.0123,174
2,0.2962,0.8787,0.0283,0.0123,102
3,0.2954,0.8739,0.0292,0.0123,153
4,0.3027,0.8860,0.0306,0.0123,178


In [7]:
def mean_sd(values):
    return f"{values.mean():.4f} +/- {values.std(ddof=1):.4f}"


print("AUC-PR:", mean_sd(rounds["auc_pr"]))
print("AUC-ROC:", mean_sd(rounds["auc_roc"]))
print("AUC-PR, ranking by ACH count:", mean_sd(rounds["auc_pr_ach_count"]))
print("positive rate:", mean_sd(rounds["positive_rate"]))
lift = (rounds["auc_pr"] / rounds["positive_rate"]).mean()
print(f"AUC-PR divided by the positive rate: {lift:.1f}")

# The declared cost of the window: positives without laundering in the window.
is_only_tail = X.index.isin(only_tail)
keep = ~is_only_tail
without = []
for k in range(N_FOLDS):
    kept = keep & (fold_of == k)
    without.append(average_precision_score(y[kept], result.scores[kept]))
print("AUC-PR without these positives:", mean_sd(pd.Series(without)))

ranks = result.scores.rank(pct=True)
print(f"median rank, positives only through tail: {ranks[is_only_tail].median():.3f}")
print(f"median rank, other positives: {ranks[(y == 1) & keep].median():.3f}")
print(f"median rank, negatives: {ranks[y == 0].median():.3f}")

AUC-PR: 0.2893 +/- 0.0218
AUC-ROC: 0.8818 +/- 0.0062
AUC-PR, ranking by ACH count: 0.0312 +/- 0.0038
positive rate: 0.0123 +/- 0.0000
AUC-PR divided by the positive rate: 23.5


AUC-PR without these positives: 0.3084 +/- 0.0238
median rank, positives only through tail: 0.652
median rank, other positives: 0.954
median rank, negatives: 0.494


In [8]:
# Sanity check: with shuffled labels, nothing can be learned honestly, so the
# AUC-PR must fall to the positive rate. A higher value would mean a bug in the loop.
shuffled = pd.Series(
    y.sample(frac=1, random_state=0).to_numpy(), index=y.index, name=y.name
)
shuffled_result = cv_predict(X, shuffled, fold_of, fit_xgboost)
shuffled_rounds = pd.Series(
    [
        average_precision_score(
            shuffled[fold_of == k], shuffled_result.scores[fold_of == k]
        )
        for k in range(N_FOLDS)
    ]
)
print("AUC-PR with shuffled labels:", mean_sd(shuffled_rounds))

AUC-PR with shuffled labels: 0.0125 +/- 0.0003


In [9]:
# Which features carry the model? Total gain, averaged over the five models.
gains = (
    pd.DataFrame(
        [m.get_booster().get_score(importance_type="total_gain") for m in models]
    )
    .reindex(columns=FEATURES)
    .fillna(0)
)
share = gains.div(gains.sum(axis=1), axis=0).mean().sort_values(ascending=False)
(100 * share).round(1).rename("gain_share_pct").to_frame()

,gain_share_pct
ach_out_tx,30.2
currency_change_tx,21.0
ach_in_tx,15.3
n_senders,6.4
in_tx,6.3
n_receivers,5.1
out_tx,3.3
active_days,2.9
cheque_tx,2.7
credit_card_tx,2.7


In [10]:
# No laundering transaction changes currency, yet this feature carries the model.
changes = (early["receiving_currency"] != early["payment_currency"]).sum()
bad = early["is_laundering"] == 1
bad_changes = (
    early.loc[bad, "receiving_currency"] != early.loc[bad, "payment_currency"]
).sum()
print(f"currency-changing tx: {changes:,}  laundering among them: {bad_changes}")
buckets = pd.cut(X["currency_change_tx"], [-1, 0, 1, 2, 5, 10, X.shape[0]])
by_bucket = y.groupby(buckets, observed=True).agg(accounts="size", positives="sum")
by_bucket["positive_rate_pct"] = (
    100 * by_bucket["positives"] / by_bucket["accounts"]
).round(2)
by_bucket

currency-changing tx: 71,779  laundering among them: 0


,accounts,positives,positive_rate_pct
currency_change_tx,,,
"(-1, 0]",505972,5069,1.00
"(0, 1]",2860,1105,38.64
"(1, 2]",1473,24,1.63
"(2, 5]",1003,40,3.99
"(5, 10]",859,40,4.66
"(10, 515078]",2911,69,2.37


In [11]:
# What is the single currency change of the accounts that have exactly one?
is_change = early["receiving_currency"] != early["payment_currency"]
changing = early.loc[is_change].assign(
    sender=lambda t: keys(t, "from"), receiver=lambda t: keys(t, "to")
)
exactly_one = X.index[X["currency_change_tx"] == 1]
sides = []
for role, column in [("sender", "sender"), ("receiver", "receiver")]:
    side = changing[changing[column].isin(exactly_one)].assign(
        account=lambda t, c=column: t[c], role=role
    )
    sides.append(side)
one_change = pd.concat(sides).drop_duplicates(["account", "timestamp"])
assert one_change["account"].is_unique and len(one_change) == len(exactly_one)
one_change["positive"] = y[one_change["account"]].to_numpy()
one_change["self_tx"] = one_change["sender"] == one_change["receiver"]

first_bad = (
    pd.concat(
        [
            early.loc[bad].assign(account=lambda t: keys(t, "from")),
            early.loc[bad].assign(account=lambda t: keys(t, "to")),
        ]
    )
    .groupby("account")["timestamp"]
    .min()
)
hours = (
    first_bad.reindex(one_change["account"]).to_numpy() - one_change["timestamp"]
).dt.total_seconds() / 3600
one_change["hours_to_first_laundering"] = hours

print(one_change.groupby("positive").size().rename("accounts").to_string())
print()
print(one_change.groupby("positive")[["self_tx"]].mean().round(3).to_string())
print()
print("role of the account, share by label:")
print(
    pd.crosstab(one_change["positive"], one_change["role"], normalize="index").round(3)
)
print()
print("payment format, share by label:")
print(
    pd.crosstab(
        one_change["payment_format"], one_change["positive"], normalize="columns"
    ).round(3)
)
print()
pos_hours = one_change.loc[one_change["positive"] == 1, "hours_to_first_laundering"]
known = pos_hours.dropna()
print(f"positives with laundering in the window: {known.size:,} of {pos_hours.size:,}")
print(f"currency change before the first laundering: {(known > 0).mean():.1%}")
print(f"hours from change to first laundering: median {known.median():.1f}")
print(
    f"  10th to 90th percentile: {known.quantile(0.1):.1f} to {known.quantile(0.9):.1f}"
)
print()
print("positive rate by account activity, among accounts with exactly one change:")
activity = pd.cut(
    (X["out_tx"] + X["in_tx"])[exactly_one], [0, 2, 5, 10, 20, 50, 100, X.shape[0]]
)
print(
    y[exactly_one]
    .groupby(activity, observed=True)
    .agg(accounts="size", positive_rate="mean")
    .round(3)
    .to_string()
)

positive
0    1755
1    1105

          self_tx
positive         
0           0.934
1           0.998

role of the account, share by label:
role      receiver  sender
positive                  
0            0.032   0.968
1            0.001   0.999

payment format, share by label:
positive            0    1
payment_format            
ACH             0.976  1.0
Wire            0.024  0.0

positives with laundering in the window: 1,105 of 1,105
currency change before the first laundering: 0.5%
hours from change to first laundering: median 0.0
  10th to 90th percentile: -46.2 to 0.0

positive rate by account activity, among accounts with exactly one change:
               accounts  positive_rate
(0, 2]               65          0.015
(2, 5]              204          0.402
(5, 10]             309          0.945
(10, 20]            907          0.131
(20, 50]            850          0.434
(50, 100]           389          0.442
(100, 515078]       136          0.515


In [12]:
# Ablation: how much of the score comes from these two feature groups?
ACH_FEATURES = ["ach_out_tx", "ach_in_tx"]
CURRENCY_FEATURES = ["currency_change_tx"]
variants = {
    "all 15 features": [],
    "without currency_change_tx": CURRENCY_FEATURES,
    "without ACH counts": ACH_FEATURES,
    "without both": CURRENCY_FEATURES + ACH_FEATURES,
}
ablation = {}
for name, dropped in variants.items():
    scores = cv_predict(X.drop(columns=dropped), y, fold_of, fit_xgboost).scores
    per_round = pd.Series(
        [
            average_precision_score(y[fold_of == k], scores[fold_of == k])
            for k in range(N_FOLDS)
        ]
    )
    ablation[name] = {
        "auc_pr_mean": per_round.mean(),
        "auc_pr_sd": per_round.std(ddof=1),
        "auc_pr_min": per_round.min(),
        "auc_pr_max": per_round.max(),
    }
pd.DataFrame(ablation).T.round(4)

,auc_pr_mean,auc_pr_sd,auc_pr_min,auc_pr_max
all 15 features,0.2893,0.0218,0.2507,0.3027
without currency_change_tx,0.2852,0.0207,0.2497,0.3000
without ACH counts,0.2678,0.0228,0.2279,0.2856
without both,0.2628,0.0220,0.2248,0.2811
